# Lesson 4：Tool、MCP、CLI 与 Skill

本课回答一个核心问题：**Agent 如何把模型的文本推理连接到真实能力？**

完成本课后，你应该能够：

1. 写出一个带 JSON Schema 的 Tool，并安全地分发调用；
2. 启动一个本地 MCP server，用 MCP client 完成发现与调用；
3. 理解 CLI 的参数、退出码和标准输出为什么适合自动化；
4. 创建一个包含 `SKILL.md` 的 Skill，并理解它何时被加载；
5. 用 OpenAI-compatible 环境变量运行一次真实 tool-calling loop。

> 除最后的可选模型单元外，所有示例都离线运行，不消耗 API 配额。建议依次执行 **Kernel → Restart Kernel and Run All Cells**。

## 0. 环境检查

本目录由 `uv` 管理。启动命令：

```bash
cd lesson4
uv sync
uv run jupyter lab lesson4_tools_mcp_cli_skill.ipynb
```

下面的代码同时兼容从 `lesson4/` 或仓库根目录执行 Notebook。

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

from dotenv import load_dotenv

cwd = Path.cwd()
LESSON_DIR = cwd if (cwd / 'pyproject.toml').exists() else cwd / 'lesson4'
assert (LESSON_DIR / 'pyproject.toml').exists(), '请从 lesson4 或仓库根目录启动 Notebook'
if str(LESSON_DIR) not in sys.path:
    sys.path.insert(0, str(LESSON_DIR))
load_dotenv(LESSON_DIR / '.env')

print('Python:', sys.version.split()[0])
print('Lesson directory:', LESSON_DIR.resolve())
print('API configured:', bool(os.getenv('OPENAI_API_KEY')))

Python: 3.12.12
Lesson directory: /Users/mabin/orca/projects/agent 实践教学大纲/tongagents-course/lesson4
API configured: False


## 1. 先建立一张地图

| 概念 | 它解决什么问题 | 核心载体 | 谁决定调用 | 本课示例 |
|---|---|---|---|---|
| **Tool** | 让模型执行一个具体动作 | 名称 + 描述 + JSON Schema + 函数 | 通常是模型或 Agent loop | `get_weather`、`calculate` |
| **MCP** | 用统一协议连接不同工具和数据源 | client/server 协议与 transport | MCP host/client | 本地 stdio server |
| **CLI** | 让人或程序在进程边界调用应用 | 命令、参数、stdin/stdout、退出码 | shell、脚本或 Agent | `demo_cli.py` |
| **Skill** | 复用一套任务说明、资源与脚本 | `SKILL.md` + 可选资源 | Agent 根据名称/描述加载 | `course-summary` |

它们不是四个互斥方案。常见组合是：**Skill 教 Agent 何时及如何工作，Agent 通过 MCP 找到 Tool，Tool 的底层实现可能调用 CLI。**

## 2. Tool：把 Python 函数变成模型可选择的动作

普通 Python 函数只有运行时知道怎么调用。模型还需要一个机器可读的“说明书”：工具名、用途、参数类型和必填字段。模型返回的是**调用意图**，真正执行函数的是应用程序。

典型闭环：

1. 应用把用户消息与 Tool schema 发给模型；
2. 模型返回 tool name 和 JSON arguments；
3. 应用校验并执行本地函数；
4. 应用把 tool result 送回模型；
5. 模型生成面向用户的最终回答。

这与 Anthropic Cookbook 的 tool-use 示例采用相同的教学拆分，但本课使用 OpenAI-compatible schema，并提供完全离线的分发器。

In [2]:
from tool_demo import TOOLS, calculate, dispatch_tool, get_weather

print(json.dumps(TOOLS[0], ensure_ascii=False, indent=2))

{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get sample weather for Beijing, Shanghai, or Shenzhen.",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "City name in English"
        }
      },
      "required": [
        "city"
      ],
      "additionalProperties": false
    }
  }
}


先不接模型，手动模拟一次模型返回的 tool call。这样可以独立测试 schema 之后的执行链路。

In [ ]:
simulated_tool_call = {
    'name': 'get_weather',
    'arguments': '{"city": "Shanghai"}',
}
tool_result = dispatch_tool(
    simulated_tool_call['name'], simulated_tool_call['arguments']
)
print(tool_result)

assert json.loads(tool_result)['temperature_c'] == 23
assert calculate('(8 + 4) / 2')['result'] == 6

### Tool 的安全边界

模型生成的参数是不可信输入。生产代码至少应做到：

- 用 allowlist 映射工具名，不允许模型拼接任意函数名；
- 校验 JSON 类型、长度、路径和权限；
- 对删除、付款、发送消息等副作用操作增加用户确认；
- 设置超时、并发限制与可审计日志；
- 在权限受限的 sandbox 中执行不可信代码。

本课的计算器没有使用 `eval()`，只解释有限的算术 AST。

In [ ]:
try:
    calculate("__import__('os').getcwd()")
except ValueError as error:
    print('Blocked unsafe expression:', error)

## 3. CLI：稳定的进程级接口

CLI（Command-Line Interface）并不是专门为大模型设计的。它把能力暴露成命令和参数，通过标准输出返回结果，通过退出码表示成功或失败。Agent 可以像普通自动化脚本一样调用 CLI。

好的 Agent-friendly CLI 通常具有：可发现的 `--help`、非交互模式、结构化 JSON 输出、稳定退出码，以及避免把密钥写到 stdout 的约定。

In [ ]:
def run_cli(*args: str) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        [sys.executable, str(LESSON_DIR / 'demo_cli.py'), *args],
        text=True, capture_output=True, check=False, cwd=LESSON_DIR
    )

help_result = run_cli('--help')
print(help_result.stdout)

weather_result = run_cli('weather', '--city', 'Beijing')
print('exit code:', weather_result.returncode)
print('stdout:', weather_result.stdout.strip())
assert weather_result.returncode == 0

## 4. MCP：在能力提供者与 Agent 之间建立协议

MCP（Model Context Protocol）是连接 AI 应用与外部系统的开放标准。它不是“另一个 Tool”：MCP server 可以暴露 **tools、resources 和 prompts**；MCP client 负责建立连接、发现能力并发起调用。

本例使用 `stdio` transport：client 启动一个子进程，通过标准输入/输出交换 MCP 消息。真实项目也可以使用网络 transport。

```text
Notebook (MCP client / host)
            │ initialize → list_tools → call_tool
            │ stdio
            ▼
mcp_server.py (FastMCP server) → Python functions
```

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command=sys.executable,
    args=[str((LESSON_DIR / 'mcp_server.py').resolve())],
)

async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()
        listed = await session.list_tools()
        print('Discovered:', [tool.name for tool in listed.tools])
        called = await session.call_tool('weather', {'city': 'Shenzhen'})
        print('Result:', [block.text for block in called.content if hasattr(block, 'text')])

assert {'weather', 'calculator'} <= {tool.name for tool in listed.tools}
assert called.isError is False

### 什么时候选 Tool，什么时候选 MCP？

如果能力只在一个应用内部使用，直接注册 Tool 最简单。如果多个 Agent host 需要复用同一个能力，或者需要标准化发现、连接与生命周期管理，MCP 更合适。接入 MCP 后，模型最终看到的 MCP tool 仍然常被转换为模型供应商所需的 tool schema。

## 5. Skill：把做事方法打包给 Agent

Skill 是可复用工作流的作者格式，而不是 RPC 或传输协议。按 OpenAI 官方 Codex 约定，一个 Skill 是包含 `SKILL.md` 的目录，还可以带 `scripts/`、`references/` 和 `assets/`。`SKILL.md` 顶部必须包含 `name` 和 `description`。

Codex 先只读取 name/description；显式提到 `$skill-name`，或任务匹配 description 时，再加载完整说明。这种方式称为 progressive disclosure。仓库内 Skill 可放在从当前目录到仓库根目录沿途的 `.agents/skills/` 中。

本课示例位于 `.agents/skills/course-summary/SKILL.md`。

In [ ]:
import yaml

skill_path = LESSON_DIR / '.agents/skills/course-summary/SKILL.md'
raw_skill = skill_path.read_text(encoding='utf-8')
_, frontmatter, instructions = raw_skill.split('---', maxsplit=2)
metadata = yaml.safe_load(frontmatter)

print('Discover phase:', metadata)
print('Load phase:', instructions.strip())
assert metadata['name'] == 'course-summary'

在真实 Codex 会话中，可以显式输入：

```text
$course-summary 总结 lesson4，并给我三个自测题
```

也可说“总结这节技术课程”，让 Agent 根据 description 隐式选择。description 要写清楚触发条件和边界，因为它承担路由作用。Skill 中的脚本仍需通过 Tool、shell 或其他执行环境运行，单靠 Markdown 不会自动执行动作。

## 6. 可选：连接 OpenAI-compatible 模型

把 `.env.example` 复制为 `.env`，配置：

```dotenv
OPENAI_API_KEY=your-api-key
OPENAI_BASE_URL=https://api.openai.com/v1
OPENAI_MODEL=gpt-4.1-mini
```

下面运行完整的两轮 tool-calling loop。没有配置时会安全跳过。不同兼容服务对 tool calling 的字段支持可能不同，应以服务商文档为准。

In [ ]:
from openai_tool_agent import run_tool_agent

api_key = os.getenv('OPENAI_API_KEY', '')
model = os.getenv('OPENAI_MODEL', '')
if api_key and api_key != 'your-api-key' and model:
    answer = run_tool_agent('上海示例天气是多少？再计算 23 * 2。')
    print(answer)
else:
    print('Skipped: configure OPENAI_API_KEY and OPENAI_MODEL in .env to run this cell.')

## 7. 总结与练习

记住四句话：

- **Tool 是一个动作的结构化契约。**
- **MCP 是发现和调用外部能力的开放协议。**
- **CLI 是稳定、可组合的进程入口。**
- **Skill 是 Agent 按需加载的做事方法与资源包。**

练习：

1. 给 `tool_demo.py` 增加 `convert_temperature` Tool，并补一个测试。
2. 在 `mcp_server.py` 暴露同一能力，重新运行 MCP 单元观察发现结果。
3. 给 CLI 增加 `--format json|text` 参数，并验证退出码。
4. 修改 Skill description，让它只在用户明确要求“自测题”时触发。

参考资料：

- [Claude Cookbooks: Tool use](https://github.com/anthropics/claude-cookbooks/tree/main/tool_use)
- [Model Context Protocol: Introduction](https://modelcontextprotocol.io/docs/getting-started/intro)
- [OpenAI 官方文档：Build skills](https://developers.openai.com/codex/skills/)